In [1]:
import os
import nemo.collections.tts as nemo_tts
import IPython.display as ipd
# Load mel spectrogram generator
spec_generator = nemo_tts.models.FastPitchModel.from_pretrained("tts_en_fastpitch_ipa").eval()
# Load vocoder
vocoder = nemo_tts.models.HifiGanModel.from_pretrained(model_name="tts_en_hifigan").eval()

[NeMo W 2026-04-11 16:42:24 <frozen importlib:241] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


[NeMo I 2026-04-11 16:42:27 common:935] Found existing object /home/dzejn/.cache/torch/NeMo/NeMo_2.7.2/tts_en_fastpitch_align_ipa/63969a9d8c2d5f4870d16fbf8064a5e0/tts_en_fastpitch_align_ipa.nemo.
[NeMo I 2026-04-11 16:42:27 common:935] Re-using file from: /home/dzejn/.cache/torch/NeMo/NeMo_2.7.2/tts_en_fastpitch_align_ipa/63969a9d8c2d5f4870d16fbf8064a5e0/tts_en_fastpitch_align_ipa.nemo
[NeMo I 2026-04-11 16:42:27 common:869] Instantiating model from pre-trained checkpoint


 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
[NeMo W 2026-04-11 16:42:46 fastpitch:214] Function ``g2p_backward_compatible_support`` is deprecated. But it will not be removed until a further notice. G2P object root directory `nemo_text_processing.g2p` has been replaced with `nemo.collections.tts.g2p`. Please use the latter instead as of NeMo 1.18.0.
[NeMo W 2026-04-11 16:42:47 _instantiate2:92] apply_to_oov_word=None, This means that some of words will remain unchanged if they are not handled by any of the rules in self.parse_one_word(). This may be intended if phonemes and chars are both valid inputs, otherwise, you may see unexpected deletions in your input.
[NeMo W 2026-04-11 16:42:47 fastpitch:119] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    dataset:
      _target_: nemo.collections.tts.torch.data.TTSDataset
    

[NeMo I 2026-04-11 16:42:48 modelPT:501] Model FastPitchModel was successfully restored from /home/dzejn/.cache/torch/NeMo/NeMo_2.7.2/tts_en_fastpitch_align_ipa/63969a9d8c2d5f4870d16fbf8064a5e0/tts_en_fastpitch_align_ipa.nemo.
[NeMo I 2026-04-11 16:42:48 common:935] Found existing object /home/dzejn/.cache/torch/NeMo/NeMo_2.7.2/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo.
[NeMo I 2026-04-11 16:42:48 common:935] Re-using file from: /home/dzejn/.cache/torch/NeMo/NeMo_2.7.2/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo
[NeMo I 2026-04-11 16:42:48 common:869] Instantiating model from pre-trained checkpoint


[NeMo W 2026-04-11 16:42:51 hifigan:54] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    dataset:
      _target_: nemo.collections.tts.data.datalayers.MelAudioDataset
      manifest_filepath: /home/fkreuk/data/train_finetune.txt
      min_duration: 0.75
      n_segments: 8192
    dataloader_params:
      drop_last: false
      shuffle: true
      batch_size: 64
      num_workers: 4
    
[NeMo W 2026-04-11 16:42:51 hifigan:54] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    dataset:
      _target_: nemo.collections.tts.data.datalayers.MelAudioDataset
      manifest_filepath: /home/fkreuk/data/val_finetune.txt
      min_duration: 3
      n_segments: 66150
  

[NeMo I 2026-04-11 16:42:52 modelPT:501] Model HifiGanModel was successfully restored from /home/dzejn/.cache/torch/NeMo/NeMo_2.7.2/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo.


In [2]:
# helper functions

def generate_audio(input_text):
    # parse() sets phoneme probability to 1, i.e. dictionary phoneme transcriptions are used for known words
    parsed = spec_generator.parse(input_text)
    spectrogram = spec_generator.generate_spectrogram(tokens=parsed)
    audio = vocoder.convert_spectrogram_to_audio(spec=spectrogram)
    display(ipd.Audio(audio.detach().to('cpu').numpy(), rate=22050))

def display_postprocessed_text(text):
    # to use dictionary entries for known words, not needed for generate_audio() as parse() handles this
    spec_generator.vocab.phoneme_probability = 1
    spec_generator.vocab.g2p.phoneme_probability = 1
    print(f"Input before tokenization: {' '.join(spec_generator.vocab.g2p(text))}\n")

In [3]:
text = "Now it is working."
generate_audio(text)